In [0]:
%sh
pwd


In [0]:
%sh
# Target your private user space inside the cluster driver
rm -rf /Workspace/Users/<your-email-address>/device_data
mkdir -p /Workspace/Users/<your-email-address>/device_data

# Pull the JSON down directly
wget -O /Workspace/Users/kiran.avulasetty@outlook.com/device_data/devices1.json https://raw.githubusercontent.com/Kiran-255666/datasets/refs/heads/main/devices1.json

#Replace ***your-email-address*** with your email address

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a stream that reads data from the folder, using a JSON schema
inputPath = '/Workspace/Users/<your-email-address>/device_data/'
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 2).json(inputPath)
print("Source stream created...")

In [0]:
# Write the stream to a delta table
delta_stream_table_path = '/Workspace/Users/<your-email-address>/delta/iotdevicedata'
checkpointpath = '/Workspace/Users/<your-email-address>/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

In [0]:
# Read the data in delta format into a dataframe
df = spark.read.format("delta").load(delta_stream_table_path)
display(df)

In [0]:
# 1. Load the underlying data stream into a dataframe
df = spark.read.format("delta").load(delta_stream_table_path)

# 2. Save it directly to your active Catalog and Schema as a native table
# Format: catalog_name.schema_name.table_name
df.write.format("delta").mode("overwrite").saveAsTable("pikapika.default.IotDeviceData")

# 3. Test and display the new table instantly using standard SQL
display(spark.sql("SELECT * FROM pikapika.default.IotDeviceData LIMIT 5"))


In [0]:
%sql
SELECT * FROM IotDeviceData;

In [0]:
%sh
wget -O /Workspace/Users/kiran.avulasetty@outlook.com/device_data/devices2.json https://raw.githubusercontent.com/Kiran-255666/datasets/refs/heads/main/devices2.json


In [0]:
%sql
SELECT * FROM IotDeviceData;